# 1. Import Libraries and Dependencies

Import the libraries required for data analysis, preprocessing, visualization, and credit risk modeling.

**Libraries:**
- NumPy
- SciPy
- Pandas
- Scikit-learn
- Matplotlib
- Seaborn
- Pickle
- ??? more other lis to e add

**Objective:** Set up the required Python libraries for the project.

In [ ]:
import numpy as np
import scipy
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# ??? comple this list and the markdown list
# sort this libs list.

# 2. Load Dataset and Initial Data Exploration

Load the LendingClub loan dataset and perform an initial exploration of its structure and characteristics.

**Exploration activities:**
- Dataset dimensions
- Initial and final observations
- Random sample observations
- Dataset structure and data types
- Numerical variables analysis
- Categorical variables analysis
- Feature and column identification

**Objective:** Understand the dataset structure, available features, and initial data quality.

In [ ]:
df = pd.read_csv(r"C:\Users\MohammadAmin\Desktop\projects\credit-risk-modeling\data\raw\loan_data_2007_2014.csv")


In [ ]:
df.shape


In [ ]:
pd.set_option('display.max_columns', None)
df.head()


In [ ]:
df.tail()


In [ ]:
df.sample(3, random_state=42)


In [ ]:
df.info()


In [ ]:
df.describe(include=["number"]).T


In [ ]:
df.describe(include=["object"]).T


In [ ]:
df.columns.tolist()


# 3. Data Cleaning and Feature Transformation

Clean and standardize existing variables and transform them into numerical and date-based features.

**Cleaning and transformation steps:**

- **`emp_length_int`:** Clean and convert employment length into a numerical value (years).
- **`earliest_cr_line_date`:** Convert the earliest credit line into datetime format.
- **`months_since_earliest_cr_line`:** Calculate the number of months since the earliest credit line using a fixed reference date.
- **Negative Duration Treatment:** Identify negative credit history durations and replace them with the maximum valid duration.
- **`term_int`:** Clean and convert the loan term into a numerical value (months).
- **`issue_d_date`:** Convert the loan issue date into datetime format.
- **`mths_since_issue_d`:** Calculate the number of months since loan issuance using the fixed reference date.

**Objective:** Improve data consistency, handle invalid values, and prepare existing variables for further preprocessing and credit risk modeling.

In [ ]:
# Work on a cleaned string representation of employment length.
employment_length = (
    df["emp_length"]
    .astype("string")
    .str.replace("+ years", "", regex=False)
    .str.replace("< 1 year", "0", regex=False)
    .str.replace("n/a", "0", regex=False)
    .str.replace(" years", "", regex=False)
    .str.replace(" year", "", regex=False)
    .str.strip()
)

# Convert the cleaned values to numbers; unparseable values become NaN.
df["emp_length_int"] = pd.to_numeric(
    employment_length,
    errors="coerce"
)

# Inspect the original and engineered values together.
df[["emp_length", "emp_length_int"]].sample(5, random_state=42)


In [ ]:
df["earliest_cr_line_date"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%y",
    errors="coerce"
)

# Check the raw value alongside its parsed datetime value.
df[["earliest_cr_line", "earliest_cr_line_date"]].sample(5, random_state=42)


In [ ]:
# Fixed observation/reference date used for this project version.
REFERENCE_DATE = pd.Timestamp("2017-12-01")

credit_line_date = df["earliest_cr_line_date"]

df["months_since_earliest_cr_line"] = (
    (REFERENCE_DATE.year - credit_line_date.dt.year) * 12
    + (REFERENCE_DATE.month - credit_line_date.dt.month)
)

df[["earliest_cr_line_date", "months_since_earliest_cr_line"]].sample(5, random_state=42)


In [ ]:
# Count negative durations before treatment
negative_duration_count = (
    df["months_since_earliest_cr_line"] < 0
).sum()

# Find the maximum non-negative duration
max_credit_history_months = df.loc[
    df["months_since_earliest_cr_line"] >= 0,
    "months_since_earliest_cr_line"
].max()

# Replace negative durations with the maximum valid duration
df.loc[
    df["months_since_earliest_cr_line"] < 0,
    "months_since_earliest_cr_line"
] = max_credit_history_months

# Print both results together
print(
    f"Negative durations before treatment: {negative_duration_count:,} | "
    f"Replacement value (maximum valid duration): {max_credit_history_months}"
)


In [ ]:
df["term_int"] = (
    df["term"]
    .str.replace(" months", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
    .astype("Int64")
)

df[["term", "term_int"]].sample(5, random_state=42)


In [ ]:
df["issue_d_date"] = pd.to_datetime(
    df["issue_d"],
    format="%b-%y",
    errors="coerce"
)

df["mths_since_issue_d"] = (
    (REFERENCE_DATE.year - df["issue_d_date"].dt.year) * 12
    + (REFERENCE_DATE.month - df["issue_d_date"].dt.month)
)

df[["issue_d", "issue_d_date", "mths_since_issue_d"]].sample(5, random_state=42)


# 4. Missing Values and Data Quality Assessment

Examine the dataset for duplicate records and missing values, assess the extent of missing data, and apply initial missing-value treatments.

**Cleaning and assessment steps:**

- **Duplicate Records:** Identify duplicate observations.
- **Missing Values:** Calculate missing-value counts and percentages across columns.
- **Missing Columns:** Identify variables containing missing values.
- **Missing Values Visualization:** Visualize the percentage of missing values to assess their distribution.
- **`total_rev_hi_lim`:** Replace missing values using `funded_amnt`.
- **`emp_title`:** Replace missing values with `'Unknown'`.

**Objective:** Assess data quality, understand missing-value patterns, and apply initial treatments to improve dataset completeness and consistency.

In [ ]:
df.duplicated().sum()


In [ ]:
# Missing values per column
missing_counts = df.isna().sum()

# Columns containing missing values
missing_cols = missing_counts[missing_counts > 0]

# Display missing values per column
display(missing_cols)

# Number of columns containing missing values
print(f"Number of columns with missing values: {len(missing_cols)}")

# Total missing values in the entire DataFrame
print(f"Total missing values: {missing_counts.sum()}")


In [ ]:
# Calculate missing values for each column
null_summary = pd.DataFrame({
    'Missing_Values': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / df.shape[0]) * 100
})

# Sort columns by missing percentage (highest first)
null_summary = null_summary.sort_values(
    by='Missing_Percentage',
    ascending=False
)

# plot:
null_summary_nonzero = null_summary[
    null_summary['Missing_Values'] > 0
]

plt.figure(figsize=(12, 12))

ax = sns.barplot(
    x=null_summary_nonzero['Missing_Percentage'],
    y=null_summary_nonzero.index
)

ax.bar_label(
    ax.containers[0],
    fmt='%.2f%%',
    padding=3,
    fontsize=9
)

plt.title('Percentage of Missing Values in Columns with Missing Data')
plt.xlabel('Missing Values (%)')
plt.ylabel('Columns')
plt.xlim(0, null_summary_nonzero['Missing_Percentage'].max() * 1.12)
plt.tight_layout()
plt.show()


In [ ]:
# ???

In [ ]:
# retain information about which applicants originally had missing income data

# df_processed['annual_inc_missing'] = (
#     df_processed['annual_inc'].isnull().astype(int)
# )

# This indicator is 1 when annual_inc is missing and 0 otherwise.

# (null_summary.index != 'annual_inc') # excluding selected columns by user 'annual_inc'

# In credit risk datasets, missing information can sometimes be informative.
# For example, applicants with missing income information might have a different default risk from applicants who reported their income.

# in this level of this project we drop also the annual_inc null rows with the 5% bellow code.

# ???

In [ ]:
# Missing values before filling
print(f"Missing values before: {df['total_rev_hi_lim'].isna().sum()}")

# Fill missing values
df['total_rev_hi_lim'] = df['total_rev_hi_lim'].fillna(df['funded_amnt'])

# Missing values after filling
print(f"Missing values after: {df['total_rev_hi_lim'].isna().sum()}")


In [ ]:
# Missing values before filling
print(f"Missing values before: {df['emp_title'].isna().sum()}")

# Fill missing values
df['emp_title'] = df['emp_title'].fillna('Unknown')

# Missing values after filling
print(f"Missing values after: {df['emp_title'].isna().sum()}")


# 5. Dependent (Target) Variable Definition – Good/Bad Classification

Convert loan repayment statuses into a binary target variable (`good_bad`) to distinguish between good and bad borrowers.

**Loan Status Mapping:**

| Loan Status | `good_bad` |
|---|---:|
| Fully Paid | 1 |
| Does not meet the credit policy. Status:Fully Paid | 1 |
| In Grace Period | 1 |
| Current | 1 |
| Charged Off | 0 |
| Default | 0 |
| Does not meet the credit policy. Status:Charged Off | 0 |
| Late (31-120 days) | 0 |
| Late (16-30 days) | 0 |

**Objective:** Establish a binary target variable representing borrower credit performance for subsequent credit risk modeling.

**Target Convention:**
- `1` = Good borrower (non-default).
- `0` = Bad borrower (default).
- `NaN` = Unmapped loan status.

In [ ]:
df["loan_status"].value_counts(dropna=False)


In [ ]:

# ============================================================
# Create Target Variable: good_bad
# ============================================================

# 1. Define loan status mapping
status_mapping = {
    "Fully Paid": 1,
    "Does not meet the credit policy. Status:Fully Paid": 1,
    "In Grace Period": 1,
    "Current": 1,
    
    "Charged Off": 0,
    "Default": 0,
    "Does not meet the credit policy. Status:Charged Off": 0,
    "Late (16-30 days)": 0,
    "Late (31-120 days)": 0
}

# 2. Create target variable
df["good_bad"] = df["loan_status"].map(status_mapping)

# 3. Identify unmapped statuses
unmapped_statuses = df.loc[
    df["good_bad"].isna(), "loan_status"
].value_counts(dropna=False)

# ============================================================
# Target Variable Summary
# ============================================================

print("=" * 70)
print("TARGET VARIABLE: good_bad")
print("=" * 70)

print(f"{'Data type:':<40} {df['good_bad'].dtype}")
print(f"{'Total observations:':<40} {len(df):,}")
print(f"{'Non-missing observations:':<40} {df['good_bad'].count():,}")
print(f"{'Missing observations:':<40} {df['good_bad'].isna().sum():,}")
print(f"{'Unique values:':<40} {df['good_bad'].nunique()}")

# ============================================================
# Unmapped Loan Statuses
# ============================================================

print("\n" + "=" * 70)
print("UNMAPPED LOAN STATUSES")
print("=" * 70)

if unmapped_statuses.empty:
    print("No unmapped loan statuses.")
else:
    display(unmapped_statuses.rename("Count").to_frame())

# ============================================================
# Loan Status Summary: Counts and Percentages
# ============================================================

print("\n" + "=" * 70)
print("LOAN STATUS SUMMARY")
print("=" * 70)

# 1. Calculate counts
loan_status_counts = pd.crosstab(
    df["loan_status"],
    df["good_bad"],
    margins=True,
    dropna=False
)

# 2. Calculate overall percentages
loan_status_percentages = (
    loan_status_counts["All"] / len(df) * 100
).round(2)

# 3. Combine selected columns
loan_status_summary = pd.concat(
    [
        loan_status_counts[[0, 1]].rename_axis(columns=None),
        loan_status_percentages.rename("Percentage (%)")
    ],
    axis=1
)

# 4. Add overall percentage distribution of good and bad loans
loan_distribution = (
    df["good_bad"].value_counts(normalize=True) * 100
).reindex([0, 1]).round(2)

loan_status_summary.loc["Class Distribution (%)"] = [
    loan_distribution[0],
    loan_distribution[1],
    100.00
]

# 5. Display final table
display(loan_status_summary)

# ============================================================
# Sample
# ============================================================

display(df[["loan_status", "good_bad"]].sample(5, random_state=42))


# 6. Train-Test Split

Separate the dataset into independent features (`X`) and the target variable (`y`), then divide the data into training and testing sets.

**Data splitting steps:**

- **Feature and Target Separation:** Define `X` as the independent variables and `y` as the target variable (`good_bad`).
- **Train-Test Split:** Divide the dataset into 80% training data and 20% testing data.
- **Randomization:** Shuffle observations and use `random_state=42` to ensure reproducibility.

**6.1. Training Dataset Inspection**

- **Combine Training Features and Target:** Combine `X_train` and `y_train` into `df_traintest`.

**6.2. Testing Dataset Inspection**

- **Combine Testing Features and Target:** Combine `X_test` and `y_test` into `df_traintest`.

**Objective:** Prepare separate training and testing datasets and inspect their target distributions before model development and evaluation.

In [ ]:
# Dependent variables 
X = df.drop("good_bad", axis=1)

# InDependent variable
y = df["good_bad"]

#Split the dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("df:", df.shape)

print("-" * 50)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("-" * 50)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


**6.1. Training Dataset Inspection**
- Combine `X_train` and `y_train` into `df_traintest`.

In [ ]:
# Combine Training Features and Target
df_traintest = X_train.copy()
df_traintest["good_bad"] = y_train

# Dataset shapes
print("df_traintest:", df_traintest.shape)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

# Target distribution
display(df_traintest["good_bad"].value_counts(dropna=False))


**6.2. Testing Dataset Inspection**
- Combine `X_test` and `y_test` into `df_traintest`.

### 6.2.Note: Apply Training data course and maping exact to the Test data
(# Run_Again)

Apply the preprocessing steps established on the training dataset to the test dataset.

**Important:** Use the exact mappings, coarse-classing rules, and category groupings defined for training data. Do not recalculate, modify, or redefine these settings using the test data. The test dataset must undergo the same transformations as the training dataset to ensure consistent feature definitions and prevent data leakage.

In [ ]:
# Run_Again

In [ ]:
# Combine Test Features and Target
# df_traintest = X_test.copy()
# df_traintest["good_bad"] = y_test

# Dataset shapes
# print("df_traintest:", df_traintest.shape)
# print("X_test:", X_test.shape)
# print("y_test:", y_test.shape)

# Target distribution
# display(df_traintest["good_bad"].value_counts(dropna=False))


# 7. WoE Analysis and Coarse Classing of Categorical Variables

Perform Weight of Evidence (WoE) analysis on categorical variables to evaluate their relationship with borrower credit performance, then group categories with similar risk patterns into fewer and more meaningful categories.

**Analysis and coarse classing steps:**

- **Target Filtering:** Include only good (`1`) and bad (`0`) borrowers in the analysis.
- **Categorical Features Selection:** Select the following variables for analysis and coarse classing:
  - `home_ownership`
  - `verification_status`
  - `purpose`
  - `initial_list_status`
  - `grade`
  - `sub_grade`
  - `addr_state`
- **WoE Calculation:** Calculate good and bad distributions and corresponding WoE values for each category.
- **Default Rate Calculation:** Calculate the proportion of bad borrowers within each category.
- **Results Visualization:** Generate plots to examine WoE values and default rates across categories.
- **Coarse Classing:** Group categories based on their observed WoE values and default rates using category mappings.
- **WoE Comparison:** Calculate and compare WoE values and default rates before and after grouping.
- **Results Storage:** Store the WoE tables for further analysis and subsequent preprocessing.

**Objective:** Identify meaningful credit risk patterns, reduce categorical complexity, and prepare categorical variables for feature encoding and model development.

**WoE Convention:**
- `good_bad = 1`: Good borrower.
- `good_bad = 0`: Bad borrower.
- `WoE = log(good_dist / bad_dist)`.
- Negative WoE: Higher concentration of bad borrowers.
- Positive WoE: Higher concentration of good borrowers.

In [ ]:
def calculate_woe_discrete(df, categorical_column, plot=True):

    # Filter good and bad borrowers only
    df_woe = df[df["good_bad"].isin([0, 1])].copy()

    # Calculate total good and bad borrowers
    total_good = (df_woe["good_bad"] == 1).sum()
    total_bad = (df_woe["good_bad"] == 0).sum()

    # Dictionary to store WoE tables
    woe_tables = {}

    # Calculate WoE for each categorical column
    for col in categorical_column:

        # Calculate statistics for each category
        woe_table = (
            df_woe.groupby(col, observed=True)["good_bad"]
            .agg(
                total="count",
                good="sum"
            )
        )

        # Calculate bad borrowers
        woe_table["bad"] = woe_table["total"] - woe_table["good"]

        # Calculate good and bad distributions
        woe_table["good_dist"] = woe_table["good"] / total_good
        woe_table["bad_dist"] = woe_table["bad"] / total_bad

        # Calculate Weight of Evidence (WoE)
        woe_table["WoE"] = np.log(
            woe_table["good_dist"] / woe_table["bad_dist"]
        )

        # Calculate default rate
        woe_table["default_rate"] = (
            woe_table["bad"] / woe_table["total"]
        )

        # Sort results by WoE
        woe_table = woe_table.sort_values("WoE")

        # Store results
        woe_tables[col] = woe_table

        # -------------------------
        # Display WoE Analysis
        # -------------------------
        table_output = woe_table.round(4).to_string()

        # Calculate separator width dynamically
        separator_width = max(
            len(table_output.split("\n")[0]),
            len(f"WoE Analysis: {col}")
        )

        # Display results
        print("\n" + "=" * separator_width)
        print(f"WoE Analysis: {col}")
        print("=" * separator_width)
        print(table_output)

        # -------------------------
        # Plot WoE
        # -------------------------
        if plot:

            # Replace infinite WoE values for plotting only
            plot_data = woe_table.copy()

            plot_data["WoE_plot"] = plot_data["WoE"].replace(
                [np.inf, -np.inf], np.nan
            )

            # Create WoE plot
            fig, ax = plt.subplots(1, 1, figsize=(12, 5))

            # -------------------------
            # Plot WoE (Line Chart)
            # -------------------------
            ax.plot(
                plot_data.index.astype(str),
                plot_data["WoE_plot"],
                marker="o",
                linestyle="-",
                linewidth=2,
                markersize=6
            )

            # Add vertical space to prevent annotation clipping
            y_min = plot_data["WoE_plot"].min()
            y_max = plot_data["WoE_plot"].max()
            padding = (y_max - y_min) * 0.15

            ax.set_ylim(y_min - padding, y_max + padding)

            ax.set_title(f"WoE by {col}")
            ax.set_xlabel(col)
            ax.set_ylabel("WoE")

            ax.axhline(
                0,
                linestyle="--",
                linewidth=1,
                color="red"
            )

            ax.tick_params(axis="x", rotation=90)

            ax.grid(
                axis="y",
                linestyle="--",
                alpha=0.5
            )

            # Annotate WoE data points
            for x, y in enumerate(plot_data["WoE_plot"]):
                if pd.notna(y):
                    ax.annotate(
                        f"{y:.2f}",
                        (x, y),
                        textcoords="offset points",
                        xytext=(0, 8),
                        ha="center",
                        fontsize=8
                    )

            plt.tight_layout()
            plt.show()

    return woe_tables


In [ ]:
# Define categorical features list for WoE analysis
categorical_column = [
    "home_ownership",
    "verification_status",
    "purpose",
    "initial_list_status",
    "grade",
    "sub_grade",
    "addr_state"
]

# Calculate WoE and generate line charts
woe_tables = calculate_woe_discrete(df_traintest, categorical_column)

# Run_Again


In [ ]:
def coarse_classing_analysis_for_categorical_column(df, column_name, category_mapping, plot=True):

    # Apply coarse classing
    new_column_name = f"{column_name}_coarse"
    df[new_column_name] = df[column_name].map(category_mapping)

    # WoE analysis: Original vs Coarse
    columns_to_analyze = [column_name, new_column_name]

    woe_comparison = calculate_woe_discrete(
        df,
        columns_to_analyze,
        plot=plot
    )

    return woe_comparison


In [ ]:
# Coarse Classing: home_ownership
column_name = "home_ownership"

category_mapping = {
    
    # Class 1
    "OTHER": "RENT_etc",
    "NONE": "RENT_etc",
    "ANY": "RENT_etc",
    "RENT": "RENT_etc",

    # Class 2
    "OWN": "OWN",
    
    # Class 3
    "MORTGAGE": "MORTGAGE",
    
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: verification_status (do not map and coarse)
column_name = "verification_status"

category_mapping = {
    "Verified": "Verified",
    "Source Verified": "Source Verified",
    "Not Verified": "Not Verified"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: purpose
column_name = "purpose"

category_mapping = {

    # Class 1
    "small_business": "purpose_01",
    "educational": "purpose_01",

    # Class 2
    "moving": "purpose_02",

    # Class 3
    "renewable_energy": "purpose_03",
    "house": "purpose_03",
    "other": "purpose_03",
    "medical": "purpose_03",
    "wedding": "purpose_03",

    # Class 4
    "vacation": "purpose_04",
    "debt_consolidation": "purpose_04",

    # Class 5
    "home_improvement": "purpose_05",
    "major_purchase": "purpose_05",
    "car": "purpose_05",

    # Class 6
    "credit_card": "purpose_06"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: initial_list_status (do not map and coarse)
column_name = "initial_list_status"

category_mapping = {
    "f": "f",
    "w": "w"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: grade
column_name = "grade"

category_mapping = {
    "G": "grade_01",
    "F": "grade_02",
    "E": "grade_03",
    "D": "grade_04",
    "C": "grade_05",
    "B": "grade_06",
    "A": "grade_07"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: sub_grade
column_name = "sub_grade"

category_mapping = {
    
    # Class 1:
    "F5": "sub_grade_01",
    "F4": "sub_grade_01",
    "F3": "sub_grade_01",
    "F2": "sub_grade_01",
    "F1": "sub_grade_01",

    "G1": "sub_grade_01",
    "G2": "sub_grade_01",
    "G3": "sub_grade_01",
    "G4": "sub_grade_01",
    "G5": "sub_grade_01",

    # Class 2:
    "E5": "sub_grade_02",
    "E4": "sub_grade_02",
    "E3": "sub_grade_02",
    "E2": "sub_grade_02",
    "E1": "sub_grade_02",

    # Class 3:
    "D5": "sub_grade_03",
    "D4": "sub_grade_03",
    "D3": "sub_grade_03",
    "D2": "sub_grade_03",
    "D1": "sub_grade_03",

    # Class 4:
    "C5": "sub_grade_04",
    "C4": "sub_grade_04",
    "C3": "sub_grade_04",
    "C2": "sub_grade_04",
    "C1": "sub_grade_04",

    # Class 5:
    "B5": "sub_grade_05",
    "B4": "sub_grade_05",
    "B3": "sub_grade_05",
    "B2": "sub_grade_05",
    "B1": "sub_grade_05",

    # Class 6:
    "A5": "sub_grade_06",
    "A4": "sub_grade_06",
    "A3": "sub_grade_06",
    "A2": "sub_grade_06",
    "A1": "sub_grade_06"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: addr_state
column_name = "addr_state"

category_mapping = {

    # Class 1
    "NE": "addr_state_01",
    "IA": "addr_state_01",
    "NV": "addr_state_01",

    # Class 2
    "FL": "addr_state_02",
    "AL": "addr_state_02",
    "NM": "addr_state_02",
    "HI": "addr_state_02",
    "MO": "addr_state_02",
    "KY": "addr_state_02",
    "CA": "addr_state_02",

    # Class 3
    "VA": "addr_state_03",
    "NY": "addr_state_03",
    "UT": "addr_state_03",
    "NC": "addr_state_03",
    "MD": "addr_state_03",
    "OK": "addr_state_03",
    "NJ": "addr_state_03",
    "LA": "addr_state_03",
    "MI": "addr_state_03",
    "AZ": "addr_state_03",
    "MN": "addr_state_03",
    "TN": "addr_state_03",

    # Class 4
    "AR": "addr_state_04",
    "OH": "addr_state_04",
    "PA": "addr_state_04",
    "DE": "addr_state_04",
    "WA": "addr_state_04",
    "MA": "addr_state_04",

    # Class 5
    "IN": "addr_state_05",
    "GA": "addr_state_05",
    "RI": "addr_state_05",
    "OR": "addr_state_05",
    "WI": "addr_state_05",
    "SD": "addr_state_05",
    "CT": "addr_state_05",
    "KS": "addr_state_05",
    "IL": "addr_state_05",
    "TX": "addr_state_05",

    # Class 6
    "MT": "addr_state_06",
    "CO": "addr_state_06",
    "SC": "addr_state_06",
    "VT": "addr_state_06",
    "AK": "addr_state_06",

    # Class 7
    "NH": "addr_state_07",
    "WY": "addr_state_07",
    "MS": "addr_state_07",
    "WV": "addr_state_07",
    "DC": "addr_state_07",
    "ME": "addr_state_07",
    "ID": "addr_state_07"
}

woe_comparison = coarse_classing_analysis_for_categorical_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


# 7. Fine Classing and WoE Analysis and Coarse Classing of Ordered Continuous Variables

Divide continuous numerical variables into multiple intervals (fine bins) to examine their relationship with borrower credit performance.

**Fine classing and analysis steps:**

- **Fine Classing:** Divide numerical variables into 50 or 100 intervals using binning techniques.
- **Continuous Features:**
  - `mths_since_issue_d`
  - `int_rate`
  - `funded_amnt`
  - `annual_inc`
  - `mths_since_last_delinq`
- **Annual Income Filtering:** Create a separate dataset by excluding observations with annual income above a defined threshold, then perform fine classing on the filtered data.
- **Delinquency Filtering:** Create a separate dataset containing observations with available `mths_since_last_delinq` values, then perform fine classing.
- **WoE Calculation:** Calculate good and bad distributions, WoE values, and default rates for each interval.
- **WoE Visualization:** Generate plots to examine WoE patterns and default rates across ordered intervals.
- **Results Storage:** Store and combine WoE tables for further analysis and coarse classing.

**Objective:** Identify risk patterns across continuous numerical variables, evaluate their relationship with borrower credit performance, and prepare them for subsequent coarse classing.

In [ ]:
# Fine Classing: mths_since_issue_d
df_traintest["mths_since_issue_d_fine_classed"] = pd.cut(
    df_traintest["mths_since_issue_d"],
    50
)

df_traintest[["mths_since_issue_d_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: mths_since_issue_d
df_traintest["mths_since_issue_d_fine_classed"] = pd.cut(
    df_traintest["mths_since_issue_d"],
    50
)

df_traintest[["mths_since_issue_d_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: int_rate 
df_traintest["int_rate_fine_classed"] = pd.cut(
    df_traintest["int_rate"],
    50
)

df_traintest[["int_rate_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: funded_amnt
df_traintest["funded_amnt_fine_classed"] = pd.cut(
    df_traintest["funded_amnt"],
    bins=50
)

df_traintest[["funded_amnt_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: annual_inc (50 bins)
df_traintest["annual_inc_fine_classed_50"] = pd.cut(
    df_traintest["annual_inc"],
    bins=50
)

df_traintest[["annual_inc_fine_classed_50"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: annual_inc (100 bins)
df_traintest["annual_inc_fine_classed_100"] = pd.cut(
    df_traintest["annual_inc"],
    bins=100
)

df_traintest[["annual_inc_fine_classed_100"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Filter annual_inc in new dataframe (df_annual_inc)

# 1. Define threshold
annual_inc_threshold = 150_000

# 2. Filter dataset (remove missing values and income above threshold)
df_annual_inc = df_traintest.loc[
    df_traintest["annual_inc"].le(annual_inc_threshold)
].copy()

# 3. Calculate summary metrics
missing_original = df_traintest["annual_inc"].isna().sum()
above_threshold = df_traintest["annual_inc"].gt(annual_inc_threshold).sum()

expected_removed = missing_original + above_threshold
actual_removed = len(df_traintest) - len(df_annual_inc)

# 4. Display results
print("=" * 70)
print("Filter annual_inc in new dataframe (df_annual_inc)")
print("=" * 70)

print(f"{'Income threshold:':<48} ${annual_inc_threshold:>12,}")
print(f"{'Original rows:':<48} {len(df_traintest):>12,}")
print(f"{'Filtered rows:':<48} {len(df_annual_inc):>12,}")

print("-" * 70)

print(f"{'Missing annual income:':<48} {missing_original:>12,}")
print(f"{'Income above threshold:':<48} {above_threshold:>12,}")
print(f"{'Expected rows removed:':<48} {expected_removed:>12,}")
print(f"{'Actual rows removed:':<48} {actual_removed:>12,}")

print("-" * 70)

print(f"{'Missing values after filtering:':<48} {df_annual_inc['annual_inc'].isna().sum():>12,}")
print(f"{'Rows removed match expectation:':<48} {expected_removed == actual_removed}")

print("=" * 70)

display(df_annual_inc[["annual_inc"]].sample(5, random_state=42))

# Run_Again


In [ ]:
# Fine Classing: annual_inc in new dataframe (df_annual_inc)
df_annual_inc["annual_inc_newdf_fine_classed"] = pd.cut(
    df_annual_inc["annual_inc"],
    bins=50
)

df_annual_inc[["annual_inc_newdf_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Fine Classing: annual_inc (50 bins)
df_traintest["mths_since_last_delinq_fine_classed"] = pd.cut(
    df_traintest["mths_since_last_delinq"],
    bins=50
)

df_traintest[["mths_since_last_delinq_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Filter mths_since_last_delinq in new dataframe (df_mths_since_last_delinq)

# 1. Filter dataset (remove missing values)
df_mths_since_last_delinq = df_traintest.loc[
    df_traintest["mths_since_last_delinq"].notna()
].copy()

# 2. Calculate summary metrics
missing_original = df_traintest["mths_since_last_delinq"].isna().sum()
missing_filtered = df_mths_since_last_delinq["mths_since_last_delinq"].isna().sum()

rows_removed = len(df_traintest) - len(df_mths_since_last_delinq)

# 3. Display results
print("=" * 70)
print("# Filter mths_since_last_delinq in new dataframe (df_mths_since_last_delinq)")
print("=" * 70)

print(f"{'Original rows:':<48} {len(df_traintest):>12,}")
print(f"{'Filtered rows:':<48} {len(df_mths_since_last_delinq):>12,}")

print("-" * 70)

print(f"{'Missing values in original:':<48} {missing_original:>12,}")
print(f"{'Missing values after filtering:':<48} {missing_filtered:>12,}")
print(f"{'Rows removed:':<48} {rows_removed:>12,}")

print("-" * 70)

print(f"{'Filtering validation:':<48} {missing_filtered == 0 and rows_removed == missing_original}")

print("=" * 70)

display(df_mths_since_last_delinq[["mths_since_last_delinq"]].sample(5, random_state=42))

# Run_Again


In [ ]:
# Fine Classing: mths_since_last_delinq
df_mths_since_last_delinq["mths_since_last_delinq_newdf_fine_classed"] = pd.cut(
    df_mths_since_last_delinq["mths_since_last_delinq"],
    bins=50
)

df_mths_since_last_delinq[["mths_since_last_delinq_newdf_fine_classed"]].sample(5, random_state=42)

# Run_Again


In [ ]:
# WoE Analysis and plot for Ordered Continuous (Numeric) Variables (Note: sort_values)
def calculate_woe_ordered_continuous(df, ordered_numeric_column, plot=True):

    # Filter good and bad borrowers only
    df_woe = df[df["good_bad"].isin([0, 1])].copy()

    # Calculate total good and bad borrowers
    total_good = (df_woe["good_bad"] == 1).sum()
    total_bad = (df_woe["good_bad"] == 0).sum()

    # Dictionary to store WoE tables
    woe_tables = {}

    # Calculate WoE for each ordered_numeric_column
    for col in ordered_numeric_column:

        # Calculate statistics for each category
        woe_table = (
            df_woe.groupby(col, observed=True)["good_bad"]
            .agg(
                total="count",
                good="sum"
            )
        )

        # Calculate bad borrowers
        woe_table["bad"] = woe_table["total"] - woe_table["good"]

        # Calculate good and bad distributions
        woe_table["good_dist"] = woe_table["good"] / total_good
        woe_table["bad_dist"] = woe_table["bad"] / total_bad

        # Calculate Weight of Evidence (WoE)
        woe_table["WoE"] = np.log(
            woe_table["good_dist"] / woe_table["bad_dist"]
        )

        # Calculate default rate
        woe_table["default_rate"] = (
            woe_table["bad"] / woe_table["total"]
        )

        # Sort results by WoE:
        # do not need to sort for numeric values that are ordered. (Because the categories should remain in their original numerical order.)
        # woe_table = woe_table.sort_values("WoE")

        # Store results
        woe_tables[col] = woe_table

        # -------------------------
        # Display WoE Analysis
        # -------------------------
        table_output = woe_table.round(4).to_string()

        # Calculate separator width dynamically
        separator_width = max(
            len(table_output.split("\n")[0]),
            len(f"WoE Analysis: {col}")
        )

        # Display results
        print("\n" + "=" * separator_width)
        print(f"WoE Analysis: {col}")
        print("=" * separator_width)
        print(table_output)

        # -------------------------
        # Plot WoE
        # -------------------------
        if plot:

            # Replace infinite WoE values for plotting only
            plot_data = woe_table.copy()

            plot_data["WoE_plot"] = plot_data["WoE"].replace(
                [np.inf, -np.inf], np.nan
            )

            # Create WoE plot
            fig, ax = plt.subplots(1, 1, figsize=(12, 5))

            # -------------------------
            # Plot WoE (Line Chart)
            # -------------------------
            ax.plot(
                plot_data.index.astype(str),
                plot_data["WoE_plot"],
                marker="o",
                linestyle="-",
                linewidth=2,
                markersize=6
            )

            # Add vertical space to prevent annotation clipping
            y_min = plot_data["WoE_plot"].min()
            y_max = plot_data["WoE_plot"].max()
            padding = (y_max - y_min) * 0.15

            ax.set_ylim(y_min - padding, y_max + padding)

            ax.set_title(f"WoE by {col}")
            ax.set_xlabel(col)
            ax.set_ylabel("WoE")

            ax.axhline(
                0,
                linestyle="--",
                linewidth=1,
                color="red"
            )

            ax.tick_params(axis="x", rotation=90)

            ax.grid(
                axis="y",
                linestyle="--",
                alpha=0.5
            )

            # Annotate WoE data points
            for x, y in enumerate(plot_data["WoE_plot"]):
                if pd.notna(y):
                    ax.annotate(
                        f"{y:.2f}",
                        (x, y),
                        textcoords="offset points",
                        xytext=(0, 8),
                        ha="center",
                        fontsize=8
                    )

            plt.tight_layout()
            plt.show()

    return woe_tables


In [ ]:
# Define ordered continuous features available in dataframe
ordered_numeric_column = [
    "term_int",
    "emp_length_int",
    "mths_since_issue_d_fine_classed",
    "int_rate_fine_classed",
    "funded_amnt_fine_classed",
    "annual_inc_fine_classed_50",
    "annual_inc_fine_classed_100",
    "mths_since_last_delinq_fine_classed"
]

# Calculate WoE for ordered continuous features
woe_tables = calculate_woe_ordered_continuous(
    df_traintest, ordered_numeric_column
)

# Calculate WoE for ordered continuous features in another dataframe
additional_woe_tables = {
    **calculate_woe_ordered_continuous(
        df_annual_inc, ["annual_inc_newdf_fine_classed"]
    ),
    **calculate_woe_ordered_continuous(
        df_mths_since_last_delinq,
        ["mths_since_last_delinq_newdf_fine_classed"]
    ),
}

# Combine all WoE tables
woe_tables.update(additional_woe_tables)

# Run_Again


In [ ]:
def coarse_classing_analysis_for_ordered_numeric_column(df, column_name, category_mapping, plot=True):

    # Apply coarse classing
    new_column_name = f"{column_name}_coarse"
    df[new_column_name] = df[column_name].map(category_mapping)

    # WoE analysis: Original vs Coarse
    columns_to_analyze = [column_name, new_column_name]

    woe_comparison = calculate_woe_ordered_continuous(
        df,
        columns_to_analyze,
        plot=plot
    )

    return woe_comparison


In [ ]:
# Coarse Classing: term_int (do not map and coarse)
column_name = "term_int"

category_mapping = {
    36: "36-Months",
    60: "60-Months"
}

woe_comparison = coarse_classing_analysis_for_ordered_numeric_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: emp_length_int
column_name = "emp_length_int"

category_mapping = {

    # Class 1
    0: "emp_length_01",
    1: "emp_length_01",
    2: "emp_length_01",
    3: "emp_length_01",
    4: "emp_length_01",
    5: "emp_length_01",
    6: "emp_length_01",
    7: "emp_length_01",
    8: "emp_length_01",
    9: "emp_length_01",

    # Class 2
    10: "emp_length_02"
}

woe_comparison = coarse_classing_analysis_for_ordered_numeric_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: mths_since_issue_d_fine_classed
column_name = "mths_since_issue_d_fine_classed"

category_mapping = {
    interval: (
        "35.91-41.4" if interval.right <= 41.4
        else "41.4-46.8" if interval.right <= 46.8
        else "46.8-50.4" if interval.right <= 50.4
        else "50.4-57.6" if interval.right <= 57.6
        else "57.6-126.0"
    )
    for interval in df_traintest[column_name].cat.categories
}

woe_comparison = coarse_classing_analysis_for_ordered_numeric_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: funded_amnt_fine_classed
column_name = "funded_amnt_fine_classed"

category_mapping = {
    interval: (
        "465.5-15680" if interval.right <= 15680
        else "15680-35000"
    )
    for interval in df_traintest[column_name].cat.categories
}

woe_comparison = coarse_classing_analysis_for_ordered_numeric_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again

# ??? this column is deleted in the udemy coarse

In [ ]:
# Coarse Classing: int_rate_fine_classed
column_name = "int_rate_fine_classed"

category_mapping = {
    interval: (
        "5.399-7.897" if interval.right <= 7.897
        else "7.897-10.374" if interval.right <= 10.374
        else "10.374-12.438" if interval.right <= 12.438
        else "12.438-15.740" if interval.right <= 15.740
        else "15.740-18.217" if interval.right <= 18.217
        else "18.217-20.694" if interval.right <= 20.694
        else "20.694-26.060"
    )
    for interval in df_traintest[column_name].cat.categories
}

woe_comparison = coarse_classing_analysis_for_ordered_numeric_column(
    df_traintest,
    column_name,
    category_mapping
)

# Run_Again


In [ ]:
# Coarse Classing: annual_inc
column_name = "annual_inc"
original_column_name = "annual_inc_fine_classed_100" # column that initial woe analysis on that

category_mapping = {
    (1747, 34478.88): "1747-34479",
    (34478.88, 52251.36): "34479-52251",
    (52251.36, 70023.84): "52251-70024",
    (70023.84, 90758.40): "70024-90758",
    (90758.40, 108530.88): "90758-108531",
    (108530.88, 150000): "108531-150000",
    (150000, np.inf): ">150000",
    (np.nan, np.nan): "Missing"
}

# ============================================================
# 1. APPLY COARSE CLASSING
# ============================================================

new_column_name = f"{column_name}_coarse"

conditions = []
choices = []

for (lower, upper), category in category_mapping.items():

    if np.isnan(lower):
        conditions.append(df_traintest[column_name].isna())

    else:
        conditions.append(
            (df_traintest[column_name] >= lower if lower == 0
             else df_traintest[column_name] > lower) &
            (df_traintest[column_name] <= upper)
        )

    choices.append(category)

df_traintest[new_column_name] = np.select(
    conditions,
    choices,
    default="Missing"
)

# ============================================================
# 2. DISPLAY CATEGORY MAPPING
# ============================================================

print(f"\n{column_name} Mapping:")
print("=" * 60)

for (lower, upper), category in category_mapping.items():

    if np.isnan(lower):
        print(f"{'Missing':<25} -> {category}")

    elif np.isinf(upper):
        print(f"{f'> {lower:,.0f}':<25} -> {category}")

    else:
        print(f"{f'{lower:,.0f} - {upper:,.0f}':<25} -> {category}")

# ============================================================
# 3. WOE ANALYSIS: ORIGINAL VS COARSE
# ============================================================

columns_to_analyze = [
    original_column_name,
    new_column_name
]

woe_comparison = calculate_woe_ordered_continuous(
    df_traintest,
    columns_to_analyze,
    plot=True
)

# ============================================================
# 4. DISPLAY SAMPLE RESULTS
# ============================================================

df_traintest[[column_name, original_column_name, new_column_name]].sample(5, random_state=42)

# Run_Again


In [ ]:
# Coarse Classing: mths_since_last_delinq
column_name = "mths_since_last_delinq"
original_column_name = "mths_since_last_delinq_fine_classed" # column that initial woe analysis on that

category_mapping = {
    (-np.inf, 30.08): "<= 30.08",
    (30.08, 60.16): "30.08-60.16",
    (60.16, np.inf): "> 60.16",
    (np.nan, np.nan): "Missing"
}

# ============================================================
# 1. APPLY COARSE CLASSING
# ============================================================

new_column_name = f"{column_name}_coarse"

conditions = []
choices = []

for (lower, upper), category in category_mapping.items():

    if np.isnan(lower):
        conditions.append(df_traintest[column_name].isna())

    else:
        conditions.append(
            (df_traintest[column_name] >= lower if lower == 0
             else df_traintest[column_name] > lower) &
            (df_traintest[column_name] <= upper)
        )

    choices.append(category)

df_traintest[new_column_name] = np.select(
    conditions,
    choices,
    default="Missing"
)

# ============================================================
# 2. DISPLAY CATEGORY MAPPING
# ============================================================

print(f"\n{column_name} Mapping:")
print("=" * 60)

for (lower, upper), category in category_mapping.items():

    if np.isnan(lower):
        print(f"{'Missing':<25} -> {category}")

    elif np.isinf(upper):
        print(f"{f'> {lower:,.0f}':<25} -> {category}")

    else:
        print(f"{f'{lower:,.0f} - {upper:,.0f}':<25} -> {category}")

# ============================================================
# 3. WOE ANALYSIS: ORIGINAL VS COARSE
# ============================================================

columns_to_analyze = [
    original_column_name,
    new_column_name
]

woe_comparison = calculate_woe_ordered_continuous(
    df_traintest,
    columns_to_analyze,
    plot=True
)

# ============================================================
# 4. DISPLAY SAMPLE RESULTS
# ============================================================

df_traintest[[column_name, original_column_name, new_column_name]].sample(5, random_state=42)

# Run_Again


# 7. One-Hot Encoding of Coarse-Classed Train Variables

Identify the coarse-classed Categorical and Countinues Variables and convert them into numerical dummy variables using one-hot encoding.

**Encoding steps:**

- **Coarse-Classed Feature Identification:** Identify columns containing `"coarse"` in their names.
- **Feature Selection:** Use the identified coarse-classed columns as Categorical and Countinues Variables.
- **One-Hot Encoding:** Generate binary indicator variables for each category using `pd.get_dummies()`.
- **Feature Integration:** Append the generated dummy variables to the original dataset (`df`) and the training dataset (`df_traintest`).
- **Results Inspection:** Review the generated dummy columns, sample observations, and updated dataset dimensions.

**Objective:** Convert coarse-classed all variables into numerical binary features suitable for subsequent credit risk modeling.

In [ ]:
# ============================================================
# Identify Coarse-Classed Columns
# ============================================================

# 1. Identify columns containing "coarse"
coarse_classed_cols = df_traintest.columns[
    df_traintest.columns.str.contains("coarse", case=False)
].tolist()

# 2. Display results
print("=" * 70)
print("COARSE-CLASSED COLUMNS")
print("=" * 70)

print(f"Number of coarse classed columns: {len(coarse_classed_cols)}")
print("\nColumn names:")
print(*coarse_classed_cols, sep="\n")

display(df_traintest[coarse_classed_cols].sample(5, random_state=42))


In [ ]:
# ============================================================
# One-Hot Encoding (coarse_classed_cols) in (df_dummies_train)
# ============================================================

# 1. Generate dummy variables
df_dummies_train = pd.get_dummies(
    df_traintest[coarse_classed_cols],
    prefix=coarse_classed_cols,
    prefix_sep=": ",
    dtype="int8"
)

# 2. Append dummy variables to DataFrames
df = pd.concat([df, df_dummies_train], axis=1)
df_traintest = pd.concat([df_traintest, df_dummies_train], axis=1)

# 3. Display results
print("=" * 70)
print("ONE-HOT ENCODING SUMMARY")
print("=" * 70)

print(f"{'Coarse classed variables encoded:':<45} {len(coarse_classed_cols):>12,}")
print(f"{'New dummy columns created:':<45} {df_dummies_train.shape[1]:>12,}")

print("-" * 70)

print(f"{'Updated df shape:':<45} {str(df.shape):>12}")
print(f"{'Updated df_traintest shape:':<45} {str(df_traintest.shape):>12}")
print(f"{'df_dummies_train shape:':<45} {str(df_dummies_train.shape):>12}")

display(df_dummies_train.sample(5, random_state=42))


**6.2. Testing Dataset Inspection**
- Combine `X_test` and `y_test` into `df_traintest`.

In [ ]:
# Combine Test Features and Target
df_traintest = X_test.copy()
df_traintest["good_bad"] = y_test

# Dataset shapes
print("df_traintest:", df_traintest.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# Target distribution
display(df_traintest["good_bad"].value_counts(dropna=False))


## Apply Training data course and maping exact to the Test data
(# Run_Again)

Apply the preprocessing steps established on the training dataset to the test dataset.

**Important:** Use the exact mappings, coarse-classing rules, and category groupings defined for training data. Do not recalculate, modify, or redefine these settings using the test data. The test dataset must undergo the same transformations as the training dataset to ensure consistent feature definitions and prevent data leakage.

In [ ]:
# Run_Again

In [ ]:
# stop_dot
.

In [ ]:
# Note:

# run all the (# Run_Again) codes that is find in the above.

# after run the (# Run_Again) codes we keep going.


# 7. One-Hot Encoding of Coarse-Classed Test Variables

Identify the coarse-classed Categorical and Countinues Variables and convert them into numerical dummy variables using one-hot encoding.

**Encoding steps:**

- **Coarse-Classed Feature Identification:** Identify columns containing `"coarse"` in their names.
- **Feature Selection:** Use the identified coarse-classed columns as Categorical and Countinues Variables.
- **One-Hot Encoding:** Generate binary indicator variables for each category using `pd.get_dummies()`.
- **Feature Integration:** Append the generated dummy variables to the original dataset (`df`) and the training dataset (`df_traintest`).
- **Results Inspection:** Review the generated dummy columns, sample observations, and updated dataset dimensions.

**Objective:** Convert coarse-classed all variables into numerical binary features suitable for subsequent credit risk modeling.

In [ ]:
# ============================================================
# Identify Coarse-Classed Columns
# ============================================================

# 1. Identify columns containing "coarse"
coarse_classed_cols = df_traintest.columns[
    df_traintest.columns.str.contains("coarse", case=False)
].tolist()

# 2. Display results
print("=" * 70)
print("COARSE-CLASSED COLUMNS")
print("=" * 70)

print(f"Number of coarse classed columns: {len(coarse_classed_cols)}")
print("\nColumn names:")
print(*coarse_classed_cols, sep="\n")

display(df_traintest[coarse_classed_cols].sample(5, random_state=42))


In [ ]:
# ============================================================
# One-Hot Encoding (coarse_classed_cols) in (df_dummies_test)
# ============================================================

# 1. Generate dummy variables
df_dummies_test = pd.get_dummies(
    df_traintest[coarse_classed_cols],
    prefix=coarse_classed_cols,
    prefix_sep=": ",
    dtype="int8"
)

# 2. Append dummy variables to DataFrames
df = pd.concat([df, df_dummies_test], axis=1)
df_traintest = pd.concat([df_traintest, df_dummies_test], axis=1)

# 3. Display results
print("=" * 70)
print("ONE-HOT ENCODING SUMMARY")
print("=" * 70)

print(f"{'Coarse classed variables encoded:':<45} {len(coarse_classed_cols):>12,}")
print(f"{'New dummy columns created:':<45} {df_dummies_test.shape[1]:>12,}")

print("-" * 70)

print(f"{'Updated df shape:':<45} {str(df.shape):>12}")
print(f"{'Updated df_traintest shape:':<45} {str(df_traintest.shape):>12}")
print(f"{'df_dummies_test shape:':<45} {str(df_dummies_test.shape):>12}")

display(df_dummies_test.sample(5, random_state=42))


# 9. Training and Testing Dataset Preparation and Export

Prepare the final training and testing feature matrices and target variables, verify their consistency, and export them as CSV files for subsequent model development.

**Preparation and export steps:**

- **Feature Comparison:** Compare the dimensions and column names of the training and testing dummy-variable datasets.
- **Target Variable Verification:** Check that `good_bad` is not included in the dummy-variable feature datasets.
- **Dataset Definition:** Assign the final feature matrices (`X_train`, `X_test`) and target variables (`y_train`, `y_test`).
- **CSV Export:** Save the four datasets as separate CSV files in the `data/processed` directory.
- **Dataset Reloading:** Load the exported CSV files into dedicated training and testing input and target DataFrames.
- **Shape Verification:** Inspect the dimensions of the reloaded datasets.
- **Index Alignment:** Verify that the feature and target indexes match for both training and testing datasets.

**Objective:** Create, export, and validate the final training and testing datasets, ensuring consistent feature structures and correct alignment between input features and target variables before model development.

In [ ]:
# ============================================================
# Compare Training and Test Features
# ============================================================

print("df_dummies_train:", df_dummies_train.shape)
print("df_dummies_test:", df_dummies_test.shape)

print("Same columns:", df_dummies_train.columns.equals(df_dummies_test.columns))

# Check whether good_bad exists in dummy DataFrames


print("good_bad (y) in df_dummies_train:", "good_bad" in df_dummies_train.columns)
print("good_bad (y) in df_dummies_test:", "good_bad" in df_dummies_test.columns)


In [ ]:
# Define training X and y datasets
X_train = df_dummies_train.copy()
y_train = y_train.copy()

# Define test X and y datasets
X_test = df_dummies_test.copy()
y_test = y_test.copy()


In [ ]:
# stop_dot
.

In [ ]:
# Define output directory
output_dir = r"C:\Users\MohammadAmin\Desktop\projects\credit-risk-modeling\data\processed"

X_train.to_csv(output_dir + r"\X_train.csv", index=True)
y_train.to_csv(output_dir + r"\y_train.csv", index=True)

X_test.to_csv(output_dir + r"\X_test.csv", index=True)
y_test.to_csv(output_dir + r"\y_test.csv", index=True)

# Display exported files
print("Successfully exported the following CSV files:")
for file_name in ["X_train.csv", "y_train.csv", "X_test.csv", "y_test.csv"]:
    print(f" - {file_name}")

print(f"Output directory: {output_dir}")

print("-" * 70)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("-" * 70)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# ??? these 4 print be here or be in bellows code boxes.

In [ ]:
data_path = r"C:\Users\MohammadAmin\Desktop\projects\credit-risk-modeling\data\processed"

loan_data_inputs_train = pd.read_csv(
    f"{data_path}\\X_train.csv",
    index_col=0 # understands that the first column represents the DataFrame index.
)

loan_data_targets_train = pd.read_csv(
    f"{data_path}\\y_train.csv",
    index_col=0
)

loan_data_inputs_test = pd.read_csv(
    f"{data_path}\\X_test.csv",
    index_col=0
)

loan_data_targets_test = pd.read_csv(
    f"{data_path}\\y_test.csv",
    index_col=0
)

print("Train and test datasets loaded successfully.")

print("-" * 50)
print("Training inputs:", loan_data_inputs_train.shape)
print("Training targets:", loan_data_targets_train.shape)

print("-" * 50)
print("Testing inputs:", loan_data_inputs_test.shape)
print("Testing targets:", loan_data_targets_test.shape)

print("-" * 50)
print("Training X and y indexes match:", loan_data_inputs_train.index.equals(loan_data_targets_train.index))
print("Testing X and y indexes match:", loan_data_inputs_test.index.equals(loan_data_targets_test.index))

print("-" * 50)
print("Missing Values per csv:")

print("X_train.csv:", loan_data_inputs_train.isna().sum().sum())
print("y_train.csv:", loan_data_targets_train.isna().sum().sum())
print("X_test.csv:", loan_data_inputs_test.isna().sum().sum())
print("y_test.csv:", loan_data_targets_test.isna().sum().sum())


# ???
# change all df that are deifne here to the name of the csv file names


# 10. Reference Category Selection and Final Model Input Preparation

- Reference Category
- Dummy Variable Trap

Remove one reference category from each one-hot-encoded categorical variable to avoid perfect multicollinearity (the dummy variable trap) in Logistic Regression.

**Preparation and validation steps:**

- **Reference Category Definition:** Specify one reference category for each coarse calssed variable based on the training data.
- **Reference Category Removal:** Remove the selected dummy columns from both training and testing datasets using the same reference categories.
- **Dummy Variable Trap:** Retain one fewer dummy variable per original categorical feature to avoid perfect multicollinearity when the model includes an intercept.
- **Final Model Inputs:** Create `inputs_train` and `inputs_test` by excluding the reference categories.
- **Feature Consistency:** Verify that training and testing inputs have identical columns and ordering.
- **Index Alignment:** Confirm that the final model input indexes match their corresponding target variable indexes.

**Important:** Apply exactly the same reference categories to both datasets. Do not select or remove reference categories independently from the test data.

**Objective:** Prepare consistent final model input matrices while avoiding perfect multicollinearity and preserving alignment between features and target variables.

In [ ]:
# ============================================================
# Define and drop Reference Categories
# ============================================================

# 1. Define Reference Categories to drop them
reference_categories = [
    "home_ownership_coarse: MORTGAGE",
    "verification_status_coarse: Not Verified",
    "purpose_coarse: purpose_01",
    "initial_list_status_coarse: f",
    "grade_coarse: grade_01",
    "sub_grade_coarse: sub_grade_01",
    "addr_state_coarse: addr_state_01",
    "term_int_coarse: 36-Months",
    "emp_length_int_coarse: emp_length_01",
    "mths_since_issue_d_fine_classed_coarse: 35.91-41.4",
    "funded_amnt_fine_classed_coarse: 15680-35000",
    "int_rate_fine_classed_coarse: 10.374-12.438",
    "annual_inc_coarse: 108531-150000",
    "mths_since_last_delinq_coarse: 30.08-60.16"
]

# 2. Remove reference categories from training and test inputs
inputs_train = loan_data_inputs_train.drop(
    columns=reference_categories
)

inputs_test = loan_data_inputs_test.drop(
    columns=reference_categories
)


# 3. Display removed reference categories in a readable table
reference_info = pd.DataFrame({
    "Original Variable": [
        category.split(":")[0]
        for category in reference_categories
    ],
    "Reference Category": [
        category.split(": ", 1)[1]
        for category in reference_categories
    ]
})

print("=" * 80)
print("Reference Categories Removed")
print("=" * 80)

display(reference_info)

print(f"\nTotal columns dropped: {len(reference_categories)}")


# 4. Validate final model inputs
print("\n" + "=" * 80)
print("Final Model Inputs Validation")
print("=" * 80)

print(f"inputs_train shape: {inputs_train.shape}")
print(f"inputs_test shape:  {inputs_test.shape}")

print("Same columns:", inputs_train.columns.equals(inputs_test.columns))


# ??? also here df names change for better understand, in drop (remove) part of the code.

In [ ]:
# Prepare the training data
X_train = inputs_train
y_train = loan_data_targets_train.squeeze() # Convert the target from (2D DataFrame) ---> (1D Series)

print("X and y indexes match:", X_train.index.equals(y_train.index))
print("Target contains no missing values:", not y_train.isna().any())
print("Target contains both classes (0 and 1):", set(y_train.unique()) == {0, 1})


In [ ]:
import numpy as np
from sklearn import linear_model
import scipy.stats as stat

# LogisticRegression with p_values
class LogisticRegression:

    def __init__(self, *args, **kwargs):
        self.model = linear_model.LogisticRegression(*args, **kwargs)

    def fit(self, X, y):

        # Fit Logistic Regression model
        self.model.fit(X, y)

        # Calculate denominator
        denom = (2.0 * (1.0 + np.cosh(
            self.model.decision_function(X)
        )))

        denom = np.tile(denom, (X.shape[1], 1)).T

        # Calculate Fisher Information Matrix
        F_ij = np.dot((X / denom).T, X)

        # Calculate covariance matrix
        Cramer_Rao = np.linalg.inv(F_ij)

        # Calculate standard errors
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))

        # Calculate z-scores
        z_scores = self.model.coef_[0] / sigma_estimates

        # Calculate p-values
        p_values = [
            stat.norm.sf(abs(x)) * 2
            for x in z_scores
        ]

        # Store model results
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values

        return self

In [ ]:
reg = LogisticRegression()

reg.fit(
    X_train,
    y_train
)


In [ ]:
# ============================================================
# Create the summary table
# ============================================================

feature_names = X_train.columns.tolist()

coefficients = reg.coef_[0]
intercept = reg.intercept_[0]

odds_ratios = np.exp(coefficients)
intercept_odds_ratio = np.exp(intercept)

p_values = np.append(np.nan, np.array(reg.p_values))

# Extract original variable names
original_variables = [
    np.nan if feature == "Intercept" else feature.split(":")[0]
    for feature in ["Intercept"] + feature_names
]

summary_table = pd.DataFrame({
    "Feature name": ["Intercept"] + feature_names,
    "Original Variable": original_variables,
    "Coefficients": [intercept] + coefficients.tolist(),
    "Odds Ratio": [intercept_odds_ratio] + odds_ratios.tolist(),
    "p_values": p_values
})

# Add statistical significance columns
summary_table["p < 0.01"] = (summary_table["p_values"] < 0.01).astype("Int64")
summary_table["p < 0.05"] = (summary_table["p_values"] < 0.05).astype("Int64")
summary_table["p < 0.10"] = (summary_table["p_values"] < 0.10).astype("Int64")
summary_table["p >= 0.10"] = (summary_table["p_values"] >= 0.10).astype("Int64")

summary_table



# interpret the value in the rows 0 and 1 and 5 for the exapmle in the project
# help with chatgpt
# ???


In [ ]:
PD = 1 - reg.predict_proba(X_train)[:, 1]

In [ ]:
The model can estimate a coefficient for every dummy variable.

Having a coefficient does not automatically mean that the variable is useful.

These coefficients describe relationships with default probability, but we still need to determine whether the estimated relationships are statistically reliable.

Statistical significance helps us determine whether the relationship estimated by a model is sufficiently strong relative to its uncertainty.




Null hypothesis
\[
H_0:\beta_j=0
\]

This means that, after accounting for the other variables in the model, the coefficient is zero.

In Logistic Regression, a coefficient of zero means that the variable does not change the log-odds of default, conditional on the other included variables.




Alternative hypothesis
\[
H_1:\beta_j\neq0
\]

This means that the coefficient is different from zero.

We use a p-value to evaluate the evidence against the null hypothesis.


P-value	        Decision
\(p < 0.01\)	Significant at the 1% level
\(p < 0.05\)	Significant at the 5% level
\(p < 0.10\)	Significant at the 10% level
\(p \geq 0.10\)	Not significant at the 10% level


A p-value is not:
- The probability that the variable is useless.
- The probability that the coefficient is zero.
- The probability that the model is correct.


It is a measure of how compatible the observed data are with the null hypothesis, under the assumptions of the statistical test.

In [ ]:
LogisticRegression_with_p_values

class LogisticRegression_with_p_values(
    LogisticRegression
):
    ...
# This is called inheritance in Python.


In [ ]:
class OriginalModel:
    def fit(self, X, y):
        print("Fitting the model")


class NewModel(OriginalModel):
    pass
# The new class inherits the fit() method from the original class.

In [ ]:
model = NewModel()

model.fit(X, y)

# It works because NewModel inherited fit().

In [ ]:
# Overriding ---> means creating a method in the child class with the same name as a method in the parent class.

class OriginalModel:
    def fit(self, X, y):
        print("Original fit")


class NewModel(OriginalModel):
    def fit(self, X, y):
        print("New fit")



model = NewModel()

model.fit(X, y)



result (output): 
New fit

In [ ]:
One original categorical variable can generate many dummy variables.
We should keep all dummy variables representing an original variable if at least one of them is statistically significant.
if we remove the insignificant dummy variables of the homw ownership and kepp significant dummy variables ---> The problem is that the original Home Ownership variable is now represented by only some of its categories.
if at least one dummy variable is statistically significant, keep the complete group.

# conclusion :
If at least one dummy variable in a group is statistically significant, retain the entire group of dummy variables representing that original categorical variable.
If none of the dummy variables in the group is statistically significant, remove the entire group.

In [ ]:
\[
p<0.05
\]

Keep Grade
Keep Home Ownership
Keep Address State
Keep Verification Status
Remove Delinquency in the Last Two Years
Keep Open Accounts
Remove Public Records
Remove Total Accounts
Keep Accounts Now Delinquent
Remove Total Revolving High Limit


In [ ]:
inputs_train_with_reference_categories
inputs_train



In [ ]:
# Original dummy variables
all_dummy_variables = (
    inputs_train_with_reference_categories.columns
)

# Select only the dummy variables
# belonging to original variables
# that we want to retain.

# Then remove the reference categories.

# Finally, use the resulting columns
# to fit the new Logistic Regression model.

In [ ]:
A variable can be statistically significant but have very little practical predictive value.

The coefficient is statistically significant, but its effect may be very small.

Conversely, a variable may have a relatively large coefficient but a high p-value because its estimate is uncertain.

Therefore, p-values should not be the only criterion for evaluating a PD model.

You should also examine:
- Predictive performance
- Stability of coefficients
- Population Stability Index (PSI)
- Discrimination metrics such as ROC-AUC and Gini
- Calibration
- Business interpretability





A variable may have a high p-value because:
- Its relationship with the target is weak.
- Its information is already captured by another variable.
- It is highly correlated with other predictors.
- The sample does not provide enough information to estimate its coefficient precisely.
Therefore, removing a variable solely because its p-value is high can sometimes reduce predictive performance.